## Lets visualize the ontology tree based on siblings only 

In [7]:
#!/usr/bin/env python3
"""
Quick visualization of ontology siblings for a query cell type
"""

import re
from collections import defaultdict
from typing import List, Dict, Set
import json

class SimpleTerm:
    def __init__(self, id_: str, name: str = ""):
        self.id = id_
        self.name = name
        self.parents = []
        self.definition = ""

class OntologyVisualizer:
    def __init__(self, obo_path: str = None):
        """Load Cell Ontology"""
        print("Loading Cell Ontology...")
        self.terms = {}
        self.parse_obo(obo_path)
        print(f"✓ Loaded {len(self.terms)} terms\n")
    
    def parse_obo(self, obo_path: str):
        """Simple OBO parser"""
        with open(obo_path, 'r') as f:
            current_term = None
            
            for line in f:
                line = line.strip()
                
                if line.startswith('[Term]'):
                    if current_term and current_term.id:
                        self.terms[current_term.id] = current_term
                    current_term = SimpleTerm("")
                    
                elif current_term:
                    if line.startswith('id: '):
                        current_term.id = line[4:].strip()
                    elif line.startswith('name: '):
                        current_term.name = line[6:].strip()
                    elif line.startswith('def: '):
                        current_term.definition = line[5:].strip()
                    elif line.startswith('is_a: '):
                        parent_match = re.match(r'is_a: (CL:\d+)', line)
                        if parent_match:
                            current_term.parents.append(parent_match.group(1))
            
            # Add last term
            if current_term and current_term.id:
                self.terms[current_term.id] = current_term
    
    def is_functional_parent(self, parent_id: str) -> bool:
        """Check if parent is functional classification (not hierarchy)"""
        if parent_id not in self.terms:
            return False
        functional_keywords = ['secretory', 'signaling', 'responsive', 'active', 'capable']
        parent_name = self.terms[parent_id].name.lower()
        return any(kw in parent_name for kw in functional_keywords)
    
    def get_biological_parents(self, cl_id: str) -> List[str]:
        """Get biological parents (exclude functional)"""
        if cl_id not in self.terms:
            return []
        
        term = self.terms[cl_id]
        bio_parents = [p for p in term.parents if not self.is_functional_parent(p)]
        
        # Fallback: if no biological parents, return all parents
        if not bio_parents:
            bio_parents = term.parents
        
        return bio_parents
    
    def get_children(self, cl_id: str) -> List[str]:
        """Get direct children of a term"""
        children = []
        for tid, term in self.terms.items():
            if cl_id in term.parents:
                children.append(tid)
        return children
    
    def get_siblings(self, cl_id: str, through_parent: str = None) -> List[Dict]:
        """Get siblings through a specific parent or all biological parents"""
        if cl_id not in self.terms:
            return []
        
        term = self.terms[cl_id]
        
        if through_parent:
            parents = [through_parent]
        else:
            parents = self.get_biological_parents(cl_id)
        
        all_siblings = {}
        
        for parent_id in parents:
            if parent_id not in self.terms:
                continue
                
            parent_name = self.terms[parent_id].name
            children = self.get_children(parent_id)
            
            for child_id in children:
                if child_id != cl_id:
                    if child_id not in all_siblings:
                        all_siblings[child_id] = {
                            'id': child_id,
                            'name': self.terms[child_id].name,
                            'through_parent': parent_name,
                            'parent_id': parent_id
                        }
        
        return list(all_siblings.values())
    
    def find_term_by_name(self, query: str) -> List[str]:
        """Find terms matching the query name"""
        matches = []
        query_lower = query.lower()
        for tid, term in self.terms.items():
            if query_lower in term.name.lower():
                matches.append(tid)
        return matches
    
    def visualize_query_context(self, query: str, show_definitions: bool = False):
        """
        Visualize the ontology context for a query cell type
        
        Parameters:
        -----------
        query : str
            Cell type name or CL ID
        """
        # Find the term
        if query.startswith('CL:'):
            cl_id = query
            if cl_id not in self.terms:
                print(f"CL ID not found: {cl_id}")
                return
            term = self.terms[cl_id]
        else:
            # Search by name
            matches = self.find_term_by_name(query)
            if not matches:
                print(f"No term found for '{query}'")
                return
            
            cl_id = matches[0]
            term = self.terms[cl_id]
            
            if len(matches) > 1:
                print(f" Multiple matches found. Using: {term.name} ({cl_id})")
                other = [f'{self.terms[m].name} ({m})' for m in matches[1:4]]
                print(f"   Other matches: {other}\n")
        
        print("="*100)
        print(f"ONTOLOGY CONTEXT: {term.name} ({cl_id})")
        print("="*100)
        
        # Show definition
        if show_definitions and term.definition:
            print(f"\nDefinition:")
            print(f"   {term.definition}\n")
        
        # Get all parents
        all_parents = term.parents
        bio_parents = self.get_biological_parents(cl_id)
        
        print(f"\nPARENTS ({len(all_parents)} total):")
        print("-"*100)
        for parent_id in all_parents:
            if parent_id not in self.terms:
                continue
            parent = self.terms[parent_id]
            is_bio = parent_id in bio_parents
            marker = "BIOLOGICAL" if is_bio else "FUNCTIONAL"
            print(f"  {marker:20s} {parent.name:<50s} ({parent_id})")
        
        # Get siblings through each biological parent
        print(f"\n SIBLINGS (background candidates):")
        print("-"*100)
        
        for parent_id in bio_parents:
            if parent_id not in self.terms:
                continue
                
            parent = self.terms[parent_id]
            children = self.get_children(parent_id)
            
            print(f"\n  Through parent: {parent.name} ({parent_id})")
            print(f"  └─ {len(children)} siblings:\n")
            
            for i, child_id in enumerate(children, 1):
                child = self.terms[child_id]
                if child_id == cl_id:
                    print(f"     {i:2d}. {child.name:<60s} ({child_id}) [QUERY]")
                else:
                    print(f"     {i:2d}.    {child.name:<60s} ({child_id})")
                    
                if i >= 30 and len(children) > 30:
                    print(f"     ... and {len(children) - 30} more siblings")
                    break
        
        # Summary
        all_siblings_set = set()
        for parent_id in bio_parents:
            children = self.get_children(parent_id)
            for child_id in children:
                if child_id != cl_id:
                    all_siblings_set.add(child_id)
        
        print(f"\n SUMMARY:")
        print("-"*100)
        print(f"  Query cell type:     {term.name}")
        print(f"  CL ID:               {cl_id}")
        print(f"  Biological parents:  {len(bio_parents)}")
        print(f"  Total siblings:      {len(all_siblings_set)} (unique)")
        print(f"  Functional parents:  {len(all_parents) - len(bio_parents)} (ignored for background)")
        
        if bio_parents and bio_parents[0] in self.terms:
            parent_name = self.terms[bio_parents[0]].name
            print("\nRECOMMENDED BACKGROUND:")
            print(f"   Use all {len(all_siblings_set)} siblings as background")
            print(f"   This represents the '{parent_name}' level of classification\n")
        
        return {
            'query_id': cl_id,
            'query_name': term.name,
            'biological_parents': bio_parents,
            'sibling_ids': list(all_siblings_set)
        }
    
    def compare_queries(self, queries: List[str]):
        """Compare sibling groups for multiple queries"""
        print("\n" + "="*100)
        print("COMPARING MULTIPLE QUERIES")
        print("="*100)
        
        results = {}
        for query in queries:
            print(f"\n{'─'*100}")
            result = self.visualize_query_context(query, show_definitions=False)
            if result:
                results[query] = result
        
        # Summary comparison
        print("\n" + "="*100)
        print("COMPARISON SUMMARY")
        print("="*100)
        print(f"\n{'Query':<30s} {'Biological Parents':<20s} {'Siblings':<15s}")
        print("-"*100)
        
        for query, result in results.items():
            print(f"{result['query_name']:<30s} {len(result['biological_parents']):<20d} {len(result['sibling_ids']):<15d}")



In [12]:


viz = OntologyVisualizer(obo_path="/mnt/lscratch/users/adhal/SingleCellUtils/data/misc/cell_onto/cl.obo")

# Example 1: GABAergic neuron
print("\n" + "#"*100)
print("# EXAMPLE 1: GABAergic neuron")
print("#"*100)
result = viz.visualize_query_context("GABAergic neuron", show_definitions=True)

# Example 2: Glutamatergic neuron
print("\n\n" + "#"*100)
print("# EXAMPLE 2: Glutamatergic neuron")
print("#"*100)
viz.visualize_query_context("glutamatergic neuron")

# Example 3: Cortical interneuron (multi-parent case)
print("\n\n" + "#"*100)
print("# EXAMPLE 3: Cortical interneuron (multi-parent)")
print("#"*100)
viz.visualize_query_context("cerebral cortex pyramidal neuron")

# Example 4: Compare multiple
print("\n\n" + "#"*100)
print("# EXAMPLE 4: Comparison")
print("#"*100)
viz.compare_queries([
    "GABAergic neuron",
    "glutamatergic neuron",
    "dopaminergic neuron",
    "interneuron"
])

Loading Cell Ontology...
✓ Loaded 18558 terms


####################################################################################################
# EXAMPLE 1: GABAergic neuron
####################################################################################################
 Multiple matches found. Using: GABAergic neuron (CL:0000617)
   Other matches: ['dorso-striatal cholinergic-GABAergic neuron (CL:0020007)', 'striatal cholinergic-GABAergic neuron (CL:0020008)', 'sst GABAergic neuron of the striatum (CL:4042037)']

ONTOLOGY CONTEXT: GABAergic neuron (CL:0000617)

Definition:
   "A neuron that uses GABA as a vesicular neurotransmitter" [GOC:tfm]


PARENTS (2 total):
----------------------------------------------------------------------------------------------------
  FUNCTIONAL           secretory cell                                     (CL:0000151)
  BIOLOGICAL           neuron                                             (CL:0000540)

 SIBLINGS (background candidates):
-------

In [9]:
# Example 4: Compare subtype
print("\n\n" + "#"*100)
print("# EXAMPLE 4: Comparison")
print("#"*100)
viz.compare_queries([
    "GABAergic neuron",
    # "L2/IT neuron",
    # "dopaminergic neuron",
    # "interneuron"
])



####################################################################################################
# EXAMPLE 4: Comparison
####################################################################################################

COMPARING MULTIPLE QUERIES

────────────────────────────────────────────────────────────────────────────────────────────────────
⚠ Multiple matches found. Using: GABAergic neuron (CL:0000617)
   Other matches: ['dorso-striatal cholinergic-GABAergic neuron (CL:0020007)', 'striatal cholinergic-GABAergic neuron (CL:0020008)', 'sst GABAergic neuron of the striatum (CL:4042037)']

ONTOLOGY CONTEXT: GABAergic neuron (CL:0000617)

👪 PARENTS (2 total):
----------------------------------------------------------------------------------------------------
  ⚪ FUNCTIONAL         secretory cell                                     (CL:0000151)
  🔵 BIOLOGICAL         neuron                                             (CL:0000540)

👥 SIBLINGS (background candidates):
----------

In [14]:
# Example 4: Compare subtype
print("\n\n" + "#"*100)
print("# EXAMPLE 4: Comparison")
print("#"*100)
viz.compare_queries([
    "caudal ganglionic eminence derived cortical interneuron",
    # "L2/IT neuron",
    # "dopaminergic neuron",
    # "interneuron"
])



####################################################################################################
# EXAMPLE 4: Comparison
####################################################################################################

COMPARING MULTIPLE QUERIES

────────────────────────────────────────────────────────────────────────────────────────────────────
❌ No term found for 'caudal ganglionic eminence derived cortical interneuron'

COMPARISON SUMMARY

Query                          Biological Parents   Siblings       
----------------------------------------------------------------------------------------------------


## Testing new retrieval system

In [10]:
# import scanpy as sc 
# import pandas as pd
# import sys 
# import os 
# project_root = os.path.abspath(os.path.join(os.path.dirname('src'), '..'))
# if project_root not in sys.path:
#     sys.path.insert(0, project_root)
# import src.io.cellxgene_pp_utils as cxg_utils
# import src.io.cell_ontology_utils as co_utils
# from importlib import reload 
# reload(cxg_utils)
# reload(co_utils)

In [11]:
# # ============================================================================
# # STEP 1: Initialize retriever with ontology
# # ============================================================================
# base_dir = "/mnt/lscratch/users/adhal/SingleCellUtils/data/misc/sc_reference_metadata"
# cl_obo_path = "/mnt/lscratch/users/adhal/SingleCellUtils/data/misc/cell_onto/cl.obo"

# retriever = co_utils.SemanticReferenceRetriever(
#     base_dir=base_dir,
#     use_sapbert=True,
#     cl_obo_path=cl_obo_path  # NEW parameter
# )

# # ============================================================================
# # STEP 2: Select background automatically
# # ============================================================================
# # Simple query
# background = retriever.select_background_for_query(
#     query="GABAergic neuron",
#     organ_filter="brain",
#     min_cells=100
# )

# # OR with fuzzy matching
# background = retriever.select_background_for_query(
#     query="caudal ganglionic eminence derived cortical interneuron",  # Will use semantic search
#     organ_filter="brain"
# )

# # ============================================================================
# # STEP 3: Review results
# # ============================================================================
# print(f"\nQuery: {background['query_info']['original_query']}")
# print(f"Resolved to: {background['query_info']['resolved_name']}")
# print(f"Background types: {background['summary']['n_sibling_types_available']}")
# print(f"Total cells: {background['summary']['total_cells']:,}")

# # View available entries
# print("\nAvailable entries:")
# print(background['available_entries'][['cell_type_name', 'dataset_id', 'n_cells', 'organ']])

# # ============================================================================
# # STEP 4: Get dataset IDs for download
# # ============================================================================
# dataset_ids = retriever.get_dataset_ids_for_background(background)
# print(f"\nDatasets to download: {len(dataset_ids)}")

# # ============================================================================
# # STEP 5: Download from CellxGene (existing workflow)
# # ============================================================================
# census_utils = cxg_utils.CellxgenePpUtils(organism='homo_sapiens')

# # Download datasets
# reference_adata = census_utils.get_by_dataset_ids(
#     dataset_ids=dataset_ids,
#     get_adata=False
# )

# # # Label as background
# # reference_adata.obs['group'] = 'background'

# # print(f"\n✓ Downloaded {reference_adata.n_obs:,} cells")

## Testing full harmonizer, retrieval and ontology system

In [21]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
import os
# get project root (two levels up from this notebook)
project_root = os.path.abspath(os.path.join(os.path.dirname('src'), '..'))
# if in notebook:
# project_root = os.path.abspath('..')   # or adjust as needed

if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.io.cell_ontology_utils as co_utils
from importlib import reload
reload(co_utils)
engine = co_utils.OntologyEngine(
    metadata_dir="/mnt/lscratch/users/adhal/SingleCellUtils/data/misc/sc_reference_metadata",
    cl_obo_path="/mnt/lscratch/users/adhal/SingleCellUtils/data/misc/cell_onto/cl.obo"
)

Loading Cell Ontology...


No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


  ✓ Loaded 18558 terms
Initializing harmonization classifier...
  ✓ Classifier ready (functional parents filtered)
Loading semantic retriever...
Loading semantic search system...
  Using: SapBERT
  No CL ontology path provided - ontology features disabled
  Model loaded
  Embeddings loaded ((433, 768))
  Metadata loaded (433 cell types)
  Keyword index built (1219 unique terms)

Ready for semantic search!
  ✓ Retriever ready


In [22]:
bg = engine.select_background("GABAergic neuron", organ="brain")
dataset_ids = engine.get_dataset_ids(bg)

In [24]:
cell_types = ["neuron", "hepatocyte", "T cell", "fibroblast"]

harmonized = engine.harmonize_types(cell_types)

In [25]:
harmonized

{'neuron': 'eukaryotic cell',
 'hepatocyte': 'eukaryotic cell',
 'T cell': 'mononuclear leukocyte',
 'fibroblast': 'connective tissue cell'}

In [27]:
# ============================================================================
# TEST DATA - Your actual cell types
# ============================================================================

cell_types_to_harmonize = [
    "radial glial cell",
    "cerebral cortex pyramidal neuron",
    "glutamatergic neuron",
    "L6b glutamatergic cortical neuron",
    "mature astrocyte",
    "sst GABAergic cortical interneuron",
    "cerebral cortex neuron",
    "astrocyte",
    "neuron",
    "glial cell",
    "medial ganglionic eminence derived GABAergic cortical interneuron",
    "caudal ganglionic eminence derived cortical interneuron",
    "oligodendrocyte precursor cell",
    "central nervous system neuron",
    "midbrain dopaminergic neuron",
    "microglial cell",
    "interneuron",
    "Purkinje cell",
    "cerebellum glutamatergic neuron",
    "GABAergic interneuron",
    "pyramidal neuron",
    "GABAergic neuron",
    "cerebellar granule cell",
    "cerebellar neuron",
    "Bergmann glial cell",
    "Mueller cell",
    "amacrine cell",
    "cerebral cortex GABAergic interneuron",
    "medium spiny neuron",
    "immature astrocyte"
]

print("="*80)
print("TESTING FUNCTIONAL PARENT FILTERING")
print("="*80)

# ============================================================================
# TEST 1: Check specific problematic cases
# ============================================================================

print("\n" + "="*80)
print("TEST 1: Verify functional parents are skipped")
print("="*80)

test_cases = [
    ("neuron", "Should NOT go through 'electrically responsive cell'"),
    ("cholinergic neuron", "Should go through 'neuron', not 'secretory cell'"),
    ("GABAergic neuron", "Should stay in neuron lineage"),
    ("astrocyte", "Should go through 'glial cell'")
]

for cell_type, expected in test_cases:
    cl_id = engine.classifier._find_term_by_name(cell_type)
    if cl_id:
        path = engine.classifier.get_path_to_root(cl_id)
        path_names = [engine.classifier.terms[p]['name'] for p in path[:5]]
        
        print(f"\n{cell_type}:")
        print(f"  Expected: {expected}")
        print(f"  Path: {' → '.join(path_names)}")
        
        # Check if any functional keywords in path
        has_functional = any(
            any(kw in name.lower() for kw in engine.classifier.functional_keywords)
            for name in path_names
        )
        
        if has_functional:
            print(f"  ❌ FAILED - Still has functional parent!")
        else:
            print(f"  ✓ PASSED - No functional parents")

# ============================================================================
# TEST 2: Preview harmonization at different levels
# ============================================================================

print("\n" + "="*80)
print("TEST 2: Harmonization preview")
print("="*80)

engine.preview_harmonization(cell_types_to_harmonize[:15], max_levels=3)

# ============================================================================
# TEST 3: Compare old vs new behavior
# ============================================================================

print("\n" + "="*80)
print("TEST 3: Harmonization results (levels_up=2)")
print("="*80)

harmonized = engine.harmonize_types(cell_types_to_harmonize, levels_up=2, verbose=True)

# Group by harmonized category
from collections import defaultdict
groups = defaultdict(list)
for orig, broad in harmonized.items():
    groups[broad].append(orig)

print(f"\nResults in {len(groups)} categories:")
print("="*80)

for broad, originals in sorted(groups.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\n{broad} ({len(originals)} types):")
    for orig in sorted(originals)[:5]:
        print(f"  - {orig}")
    if len(originals) > 5:
        print(f"  ... and {len(originals)-5} more")

# ============================================================================
# TEST 4: Specific expected behaviors
# ============================================================================

print("\n" + "="*80)
print("TEST 4: Expected behaviors")
print("="*80)

expectations = {
    "All GABAergic interneurons should map to same category": [
        "sst GABAergic cortical interneuron",
        "cerebral cortex GABAergic interneuron",
        "GABAergic interneuron",
        "GABAergic neuron"
    ],
    "All astrocytes should map to same category": [
        "mature astrocyte",
        "immature astrocyte",
        "astrocyte",
        "Bergmann glial cell"
    ],
    "All neurons should NOT map to 'electrically responsive cell'": [
        "neuron",
        "pyramidal neuron",
        "interneuron",
        "cerebellar neuron"
    ]
}

for expectation, test_types in expectations.items():
    print(f"\n{expectation}:")
    
    # Get harmonized values
    harmonized_vals = []
    for ct in test_types:
        if ct in harmonized:
            harmonized_vals.append((ct, harmonized[ct]))
    
    # Check if they match
    unique_vals = set([h[1] for h in harmonized_vals])
    
    for ct, hval in harmonized_vals:
        print(f"  {ct:50s} → {hval}")
    
    # Special check for functional parents
    if "electrically responsive cell" in expectation.lower():
        has_functional = any("electrically responsive" in h[1].lower() for h in harmonized_vals)
        if has_functional:
            print("  ❌ FAILED - Still mapping to functional parent!")
        else:
            print("  ✓ PASSED - No functional parents")
    else:
        if len(unique_vals) == 1:
            print(f"  ✓ PASSED - All map to: {list(unique_vals)[0]}")
        else:
            print(f"  ⚠ INFO - Map to {len(unique_vals)} categories: {unique_vals}")

print("\n" + "="*80)
print("TESTS COMPLETE")
print("="*80)


TESTING FUNCTIONAL PARENT FILTERING

TEST 1: Verify functional parents are skipped

neuron:
  Expected: Should NOT go through 'electrically responsive cell'
  Path: neuron → neural cell → eukaryotic cell → cell
  ✓ PASSED - No functional parents

cholinergic neuron:
  Expected: Should go through 'neuron', not 'secretory cell'
  Path: cholinergic neuron → neuron → neural cell → eukaryotic cell → cell
  ✓ PASSED - No functional parents

GABAergic neuron:
  Expected: Should stay in neuron lineage
  Path: GABAergic neuron → neuron → neural cell → eukaryotic cell → cell
  ✓ PASSED - No functional parents

astrocyte:
  Expected: Should go through 'glial cell'
  Path: astrocyte → macroglial cell → glial cell → neuron associated cell → neural cell
  ✓ PASSED - No functional parents

TEST 2: Harmonization preview
HARMONIZATION PREVIEW

────────────────────────────────────────────────────────────────────────────────
LEVEL 1 (go up 1 step)
─────────────────────────────────────────────────────────